# Tutorial 05: Advanced Features

Quantum control flow, multi-bit registers, and circuit optimization.

## Quantum Superposition Control (qif)

`qif` creates coherent superposition ！ both branches happen at once.

In [ ]:
from quonic import qif, qgate, qshow, reset
from quonic.gates import H, X, Z, I

reset()
qgate(H, 0)  # Control qubit in superposition
qif(0).then(X, 1).else_(Z, 1)  # Both branches applied coherently
qshow()

## Multi-Qubit qif (v0.5.0+)

Control multi-qubit gates: controlled-CX = Toffoli, controlled-SWAP = Fredkin.

In [ ]:
from quonic.gates import CX, SWAP

# Controlled-CX = Toffoli
reset()
qgate(X, 0)
qgate(X, 1)
qif(0).then(CX, 1, 2).else_(I, 1, 2)
qshow()  # Should give |111>

## Classical Control Flow (cif)

`cif` measures first, then branches ！ classical mixed state, not entanglement.

In [ ]:
from quonic import cif, creg

reset()
qgate(H, 0)
cif(0).then(X, 1).else_(Z, 1)
qshow()  # Classical mixture: 4 basis states ~25% each

## Multi-Bit Classical Registers

```python
reg = creg("reg", width=2)
reg.measure(0, bit=0)
reg.measure(1, bit=1)
cif(reg, 2).then(X, 2).else_(I, 2)  # Branch on reg == 2
```

## Circuit Optimization (v0.5.0+)

Automatically reduce gate count and circuit depth.

In [ ]:
from quonic import optimize
from quonic.compiler import optimize_cancel, optimize_commute, optimize_peephole
from quonic.ir import GateOperation

# X，X = I (self-inverse cancellation)
reset()
qgate(X, 0)
qgate(X, 0)
circ = current_circuit()
optimized = optimize(circ, passes=("cancel",))
gate_ops = [op for op in optimized.ops if isinstance(op, GateOperation) and op.name != "measure"]
print(f"X，X ★ {len(gate_ops)} gates (should be 0)")

from quonic.stack import current_circuit

In [ ]:
# CX，CX，CX = SWAP (peephole)
from quonic.gates import CX
from quonic.stack import current_circuit

reset()
qgate(CX, 0, 1)
qgate(CX, 1, 0)
qgate(CX, 0, 1)
circ = current_circuit()
optimized = optimize(circ, passes=("peephole",))
gate_ops = [op for op in optimized.ops if isinstance(op, GateOperation) and op.name != "measure"]
print(f"CX，CX，CX ★ {[op.name for op in gate_ops]} (should be ['swap'])")

## What's Next

- Explore the [Examples](../examples.md) for more use cases
- Read the [API Reference](../api/ir.md) for complete documentation
- Check the [Benchmarks](../benchmarks.md) for performance data